In [1]:
import os
import zipfile
import rasterio
from rasterio.merge import merge
from rasterio.crs import CRS
import numpy as np
import glob
from rasterio.merge import merge
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS

In [ ]:
region = 'Thames'
year = 2019

extract_folder = f"G:\DTM_ref\Wash\Wash_combine/Zemp_extracted"
reproject_folder = f"G:\DTM_ref\Wash\Wash_combine/Zemp_proj"
os.makedirs(extract_folder, exist_ok=True)
os.makedirs(reproject_folder, exist_ok=True)

# zip_folder = f'G:\DTM_ref\{region}/{region}_{year}'
output_tiff = f'G:\DTM_ref\Wash\Wash_combine/{region}_merge.tif'
output_masked_tiff =  f'G:\DTM_ref\Wash\Wash_combine/{region}_masked_{year}.tif'
nodata_value =  -999

In [9]:
region = 'Thames'
year = 2020

extract_folder = f"G:\Refer_DEM\{region}/{region}_{year}_copy/Zemp_extracted"
reproject_folder = f"G:\Refer_DEM\{region}/{region}_{year}_copy/Zemp_proj"
# os.makedirs(extract_folder, exist_ok=True)
# os.makedirs(reproject_folder, exist_ok=True)

zip_folder = f'G:\Refer_DEM\{region}/{region}_{year}_copy'
output_tiff = f'G:\Refer_DEM\{region}/{region}_{year}_copy/{region}_merge_{year}_Z.tif'
output_masked_tiff =  f'G:\Refer_DEM\{region}/{region}_{year}_copy/{region}_masked_{year}_Z.tif'
nodata_value =  -999

In [ ]:

# #1️⃣ 解压所有 ZIP 文件
# for zip_file in glob.glob(os.path.join(zip_folder, "*.zip")):
#     with zipfile.ZipFile(zip_file, "r") as zip_ref:
#         zip_ref.extractall(extract_folder)

# 2️⃣ 读取解压后的 TIFF 影像，并转换坐标系（若需要）
tiff_files = glob.glob(os.path.join(extract_folder, "*.tif"))
reprojected_files = []

for tiff in tiff_files:
    with rasterio.open(tiff) as src:
        src_crs = src.crs
        dst_crs = CRS.from_epsg(32630)  # WGS 1984 UTM Zone 30N

        if src_crs != dst_crs:
            # 计算重投影参数
            transform, width, height = calculate_default_transform(
                src.crs, dst_crs, src.width, src.height, *src.bounds, resolution=(10, 10)
            )

            # 更新元数据
            out_meta = src.meta.copy()
            out_meta.update({
                "crs": dst_crs,
                "transform": transform,
                "width": width,
                "height": height,
                "nodata": nodata_value,  # 显式设置 nodata
                "dtype": "float32"  # 确保数据类型
            })

            # 生成新的文件路径
            reprojected_tiff = os.path.join(reproject_folder, os.path.basename(tiff))
            reprojected_files.append(reprojected_tiff)

            # 执行重投影
            with rasterio.open(reprojected_tiff, "w", **out_meta) as dst:
                for i in range(1, src.count + 1):
                    reproject(
                        source=rasterio.band(src, i),
                        destination=rasterio.band(dst, i),
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=transform,
                        dst_crs=dst_crs,
                        resampling=Resampling.nearest,
                        src_nodata=src.nodata,  # 读取源影像的 nodata
                        dst_nodata=nodata_value  # 设定目标影像的 nodata
                    )
        else:
            reprojected_files.append(tiff)  # 若已是 WGS 1984，直接使用

# 3️⃣ 影像拼接
datasets = [rasterio.open(f) for f in reprojected_files]
mosaic, out_trans = merge(datasets, nodata=nodata_value)  # 避免 nodata 警告

# 4️⃣ 保存拼接影像
out_meta = datasets[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_trans,
    "nodata": nodata_value  # 确保 nodata 统一
})

with rasterio.open(output_tiff, "w", **out_meta) as dest:
    dest.write(mosaic)

print(f"✅ 拼接完成，影像已保存至: {output_tiff}")

# 设置路径
input_path = output_tiff
output_path = output_masked_tiff  # 保存的新路径

# 打开并读取数据
with rasterio.open(input_path) as src:
    data = src.read(1)  # 读取第一波段
    meta = src.meta.copy()

# 应用条件：把小于 -5 或大于 5 的值改成 6
data[(data < -5) | (data > 5)] = 6

# 保存修改后的数据
with rasterio.open(output_path, "w", **meta) as dst:
    dst.write(data, 1)

print(f"✅ 修改后的影像已保存到: {output_path}")


✅ 拼接完成，影像已保存至: G:\DTM_ref\Wash\Wash_combine/Wash_merge.tif
✅ 修改后的影像已保存到: G:\DTM_ref\Wash\Wash_combine/Wash_masked_2022.tif


In [ ]:
for year in [2020]:

    region = 'Thames'


    extract_folder = f"G:\DTM_ref\{region}/{region}_{year}_copy/Zemp_extracted"
    reproject_folder = f"G:\DTM_ref\{region}/{region}_{year}_copy/Zemp_proj"
    # os.makedirs(extract_folder, exist_ok=True)
    # os.makedirs(reproject_folder, exist_ok=True)

    zip_folder = f'G:\DTM_ref\{region}/{region}_{year}_copy'
    output_tiff = f'G:\DTM_ref\{region}/{region}_{year}_copy/{region}_merge_{year}.tif'
    output_masked_tiff =  f'G:\DTM_ref\{region}/{region}_{year}_copy/{region}_masked_{year}.tif'
    nodata_value =  -999



    # #1️⃣ 解压所有 ZIP 文件
    # for zip_file in glob.glob(os.path.join(zip_folder, "*.zip")):
    #     with zipfile.ZipFile(zip_file, "r") as zip_ref:
    #         zip_ref.extractall(extract_folder)

    # 2️⃣ 读取解压后的 TIFF 影像，并转换坐标系（若需要）
    tiff_files = glob.glob(os.path.join(extract_folder, "*.tif"))
    reprojected_files = []

    for tiff in tiff_files:
        with rasterio.open(tiff) as src:
            src_crs = src.crs
            dst_crs = CRS.from_epsg(32631)  # WGS 1984 UTM Zone 30N

            if src_crs != dst_crs:
                # 计算重投影参数
                transform, width, height = calculate_default_transform(
                    src.crs, dst_crs, src.width, src.height, *src.bounds, resolution=(10, 10)
                )

                # 更新元数据
                out_meta = src.meta.copy()
                out_meta.update({
                    "crs": dst_crs,
                    "transform": transform,
                    "width": width,
                    "height": height,
                    "nodata": nodata_value,  # 显式设置 nodata
                    "dtype": "float32"  # 确保数据类型
                })

                # 生成新的文件路径
                reprojected_tiff = os.path.join(reproject_folder, os.path.basename(tiff))
                reprojected_files.append(reprojected_tiff)

                # 执行重投影
                with rasterio.open(reprojected_tiff, "w", **out_meta) as dst:
                    for i in range(1, src.count + 1):
                        reproject(
                            source=rasterio.band(src, i),
                            destination=rasterio.band(dst, i),
                            src_transform=src.transform,
                            src_crs=src.crs,
                            dst_transform=transform,
                            dst_crs=dst_crs,
                            resampling=Resampling.nearest,
                            src_nodata=src.nodata,  # 读取源影像的 nodata
                            dst_nodata=nodata_value  # 设定目标影像的 nodata
                        )
            else:
                reprojected_files.append(tiff)  # 若已是 WGS 1984，直接使用

    # 3️⃣ 影像拼接
    datasets = [rasterio.open(f) for f in reprojected_files]
    mosaic, out_trans = merge(datasets, nodata=nodata_value)  # 避免 nodata 警告

    # 4️⃣ 保存拼接影像
    out_meta = datasets[0].meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_trans,
        "nodata": nodata_value  # 确保 nodata 统一
    })

    with rasterio.open(output_tiff, "w", **out_meta) as dest:
        dest.write(mosaic)

    print(f"✅ 拼接完成，影像已保存至: {output_tiff}")

    # 设置路径
    input_path = output_tiff
    output_path = output_masked_tiff  # 保存的新路径

    # 打开并读取数据
    with rasterio.open(input_path) as src:
        data = src.read(1)  # 读取第一波段
        meta = src.meta.copy()

    # 应用条件：把小于 -5 或大于 5 的值改成 6
    data[(data < -5) | (data > 5)] = 6

    # 保存修改后的数据
    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(data, 1)

    print(f"✅ 修改后的影像已保存到: {output_path}")

✅ 拼接完成，影像已保存至: G:\DTM_ref\Thames/Thames_2020_copy/Thames_merge_2020.tif
✅ 修改后的影像已保存到: G:\DTM_ref\Thames/Thames_2020_copy/Thames_masked_2020.tif


In [2]:
input_path = 'G:\DTM_ref\Wash\Wash_2022\Wash_merge_2022.tif'
output_path = 'G:\DTM_ref\Wash\Wash_2022\Wash_masked_combine.tif'  # 保存的新路径

# 打开并读取数据
with rasterio.open(input_path) as src:
    data = src.read(1)  # 读取第一波段
    meta = src.meta.copy()

# 应用条件：把小于 -5 或大于 5 的值改成 6
data[(data < -5) | (data > 5)] = 6

# 保存修改后的数据
with rasterio.open(output_path, "w", **meta) as dst:
    dst.write(data, 1)

print(f"✅ 修改后的影像已保存到: {output_path}")

✅ 修改后的影像已保存到: G:\DTM_ref\Wash\Wash_2022\Wash_masked_combine.tif


In [ ]:

input_path = 'G:\DTM_ref\Wash\Wash_2022\Araw_combine.tif'
output_path = 'G:\DTM_ref\Araw\Araw_masked_combine.tif'  # 保存的新路径

# 打开并读取数据
with rasterio.open(input_path) as src:
    data = src.read(1)  # 读取第一波段
    meta = src.meta.copy()

# 应用条件：把小于 -5 或大于 5 的值改成 6
data[(data < -5) | (data > 5)] = 6

# 保存修改后的数据
with rasterio.open(output_path, "w", **meta) as dst:
    dst.write(data, 1)

print(f"✅ 修改后的影像已保存到: {output_path}")

✅ 修改后的影像已保存到: G:\DTM_ref\Araw\Araw_masked_combine.tif


In [ ]:
import rasterio
import numpy as np

input_path = r'G:\ICESat-2\Region3_map\Region3b1_map\A2019\Region3b1_2019_FB6_new9.tif'
target     = r'G:\ICESat-2\Region3_map\Region3b1_map\A2019\Region3b1_2019_FB6_new2.tif'
output     = r'G:\ICESat-2\Region3_map\Region3b1_map\A2019\Region3b1_2019_merge.tif'

# 打开 input 文件，读取所有波段
with rasterio.open(input_path) as src:
    data = src.read()  # data.shape = (bands, height, width)
    meta = src.meta.copy()

# 打开 target 文件，读取第6波段
with rasterio.open(target) as src2:
    band6 = src2.read(6)  # 注意：波段索引是从1开始

# 替换 input 的第7波段
data[6, :, :] = band6  # data[6] 对应第7波段

# 更新并输出到新文件
with rasterio.open(output, 'w', **meta) as dst:
    dst.write(data)

print("✅ 输出完成:", output)

In [24]:
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np


for year in [2019, 2020, 2021, 2022, 2023, 2024]:
    
    input_path  = f'G:\ICESat-2\Region3_map\Region3b1_map\A{year}\Region3b1_{year}_FB6_new9.tif'
    target_path = f'G:\ICESat-2\Region3_map\Region3b1_map\A{year}\Region3b1_{year}_FB6_new2.tif'
    output_path = f'G:\ICESat-2\Region3_map\Region3b1_map\A{year}\Region3b1_{year}_FB6_new9_.tif'

    # 打开 input，作为基准
    with rasterio.open(input_path) as src:
        data = src.read()          # 所有波段
        data7 = src.read(7)        # 第7波段
        meta = src.meta.copy()
        transform_in = src.transform
        crs_in = src.crs
        shape_in = data7.shape     # (height, width)

    # 打开 target，并读取第6波段
    with rasterio.open(target_path) as src2:
        mask_raw = src2.read(5)

        # 创建一个与 input 相同尺寸的空数组
        mask_aligned = np.zeros(shape_in, dtype=mask_raw.dtype)

        # 将 target 的第6波段重采样+裁剪到 input 的范围
        reproject(
            source=mask_raw,
            destination=mask_aligned,
            src_transform=src2.transform,
            src_crs=src2.crs,
            dst_transform=transform_in,
            dst_crs=crs_in,
            dst_width=shape_in[1],
            dst_height=shape_in[0],
            resampling=Resampling.nearest  # 最近邻，适合掩膜/分类数据
        )

    # 替换逻辑
    data7 = mask_aligned
    data[6, :, :] = data7   # 注意 data[6] 对应第7波段

    # 写出结果
    with rasterio.open(output_path, 'w', **meta) as dst:
        dst.write(data)

    print("✅ 已完成输出:", output_path)


✅ 已完成输出: G:\ICESat-2\Region3_map\Region3b1_map\A2019\Region3b1_2019_FB6_new9_.tif
✅ 已完成输出: G:\ICESat-2\Region3_map\Region3b1_map\A2020\Region3b1_2020_FB6_new9_.tif
✅ 已完成输出: G:\ICESat-2\Region3_map\Region3b1_map\A2021\Region3b1_2021_FB6_new9_.tif
✅ 已完成输出: G:\ICESat-2\Region3_map\Region3b1_map\A2022\Region3b1_2022_FB6_new9_.tif
✅ 已完成输出: G:\ICESat-2\Region3_map\Region3b1_map\A2023\Region3b1_2023_FB6_new9_.tif
✅ 已完成输出: G:\ICESat-2\Region3_map\Region3b1_map\A2024\Region3b1_2024_FB6_new9_.tif
